In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
spark = SparkSession.builder.appName("Gold_Layer").getOrCreate()

In [0]:
settlement_silver_df = spark.read.table("workspace.upi_schema.settlement_silver")

upi_stream_silver_view_df = spark.read.table("workspace.upi_schema.upi_stream_silver_view")

settlement_silver_df = settlement_silver_df.alias("sm")
upi_stream_silver_view_df = upi_stream_silver_view_df.alias("st")
joined_df = upi_stream_silver_view_df.join(settlement_silver_df,on = ["txn_id"],how = "fullouter").select(
        col("st.txn_id").alias("upi_txn_id"),
        col("st.amount").alias("upi_trans_amount")
        ,col("st.rrn").alias("upi_trans_rrn")
        ,col("sm.txn_id").alias("settlement_txn_id")
        ,col("sm.rrn").alias("settlement_rrn")
        ,col("*")
    ).drop("rrn").drop("amount").drop("txn_id")

In [0]:
reconciled_df = (
    joined_df
    .withColumn("reconciliation"
    ,when((col("upi_txn_id").isNotNull()) & (col("settlement_txn_id").isNotNull()) & (col("status")=="SUCCESS") & (col("settlement_status")=="SETTLED"), "RECONCILED")

    .when((col("upi_txn_id").isNotNull()) & (col("settlement_txn_id").isNotNull()) & (col("settlement_status")=="SETTLED") & (col("status")=="PENDING"),"SETTLED RECONCILED")
    
    .when((col("status")=="PENDING") & (col("settlement_status")=="PENDING") & (date_add(col("initiated_timestamp"),1) < (col("settlement_date"))), "STALE PENDING")
    .when((col("status")=="PENDING") & (col("settlement_status")=="PENDING") & (date_add(col("initiated_timestamp"),1) >= (col("settlement_date"))), "PENDING")
    .when((col("status") == "SUCCESS") & (col("settlement_status") == "PENDING"), "DISCREPANCY STATUS MISMATCH")
    .when((col("status")== "SUCCESS") & (col("settlement_txn_id").isNull()), "DISCREPANCY UNSETTLED SUCCESS")

    .when((col("status")=="PENDING") & (col("settlement_txn_id").isNull()) & (date_add(col("initiated_timestamp"),1) > current_date()),"PENDING WITHIN SLA")
    .when((col("status")=="PENDING") & (col("settlement_txn_id").isNull()) & (date_add(col("initiated_timestamp"),1) < current_date()),"DISCREPANCY STALE PENDING")

    .when((col("upi_txn_id").isNull()) & (col("settlement_txn_id").isNotNull()), "ORPHAN RECORD")
    .otherwise("UNKNOWN")
    )
    .withColumn("flag",when(col("reconciliation").contains("RECONCILED"),0).otherwise(1))
)


reconciled_df.display()


reconciled_df.write.mode("append").saveAsTable("workspace.upi_schema.reconciled")


upi_txn_id,upi_trans_amount,upi_trans_rrn,settlement_txn_id,settlement_rrn,key,payer_vpa,payee_vpa,payer_bank,payee_bank,txn_type,channel,status,npci_response_code,initiated_timestamp,topic,partition,offset,timestamp,timestampType,late_flag,txn_date,txn_hour,silver_processed_timestamp,settlement_amount,settlement_status,settlement_date,settled_timestamp,run_id,reconciliation,flag
GWYD0ILMTO,7555.67,R61477155,GWYD0ILMTO,R61477155,GWYD0ILMTO,zoe@upi,rebecca@upi,SBI,ICICI,P2P,COLLECT,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,3,9,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,7555.67,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GNLNAVNCBH,20344.52,R79557118,GNLNAVNCBH,R79557118,GNLNAVNCBH,frederick@upi,jennifer@upi,CANARA,AXIS,P2M,APP,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,0,14,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,20344.52,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GKP8GYOCNI,1030.67,R21807318,GKP8GYOCNI,R21807318,GKP8GYOCNI,alexandra@upi,christian@upi,PNB,KOTAK,P2P,COLLECT,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,1,13,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,1030.67,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GTO3BBXARY,10318.33,R51413004,GTO3BBXARY,R51413004,GTO3BBXARY,tommy@upi,sarah@upi,CANARA,ICICI,P2P,APP,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,0,1,2026-08-30T12:21:06.941Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,10318.33,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
G2FKWBR7FK,19506.78,R72221634,G2FKWBR7FK,R72221634,G2FKWBR7FK,eric@upi,brittney@upi,YES,ICICI,P2M,COLLECT,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,0,6,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,19506.78,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GSLUS9HYSG,14685.00,R49043528,GSLUS9HYSG,R49043528,GSLUS9HYSG,teresa@upi,thomas@upi,YES,PNB,P2P,COLLECT,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,1,15,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,14685.00,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GX0168Y7OM,61400.62,R38271100,GX0168Y7OM,R38271100,GX0168Y7OM,denise@upi,richard@upi,AXIS,INDUSIND,P2M,QR,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,3,14,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,61400.62,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GUQ4PZ05KZ,30596.57,R86321998,GUQ4PZ05KZ,R86321998,GUQ4PZ05KZ,christopher@upi,brian@upi,INDUSIND,AXIS,P2P,QR,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,4,17,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,30596.57,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GU7ZD3IOUX,55593.40,R89664452,GU7ZD3IOUX,R89664452,GU7ZD3IOUX,rebecca@upi,dylan@upi,HDFC,AXIS,P2P,APP,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,0,17,2026-08-30T12:21:06.943Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,55593.40,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0
GUV6QIEYSZ,56796.36,R39153691,GUV6QIEYSZ,R39153691,GUV6QIEYSZ,susan@upi,zachary@upi,KOTAK,BOB,P2M,QR,SUCCESS,00,2026-08-30T17:51:02.000Z,UPI_Transactions,0,12,2026-08-30T12:21:06.942Z,0,0,2026-08-30,17,2026-08-30T12:26:06.772Z,56796.36,SETTLED,2026-08-30,2026-08-30T12:21:35.000Z,run_20260823_130000,RECONCILED,0


> # Metrics

In [0]:
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_date", "")

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
gold_count=reconciled_df.count()
data =[run_id,run_date,gold_count]
cols=["run_id","run_date","gold_count"]

gold_count_df = spark.createDataFrame([data],cols)
gold_count_df.createOrReplaceTempView("gold_count_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING gold_count_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.gold_count=s.gold_count
          WHEN NOT MATCHED then INSERT (run_id,run_date,gold_count) VALUES(s.run_id,s.run_date,s.gold_count)
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
gold_discrep_unsettled_success_count=reconciled_df.filter(col("reconciliation")=="DISCREPANCY UNSETTLED SUCCESS").count()
data =[run_id,run_date,gold_discrep_unsettled_success_count]
cols=["run_id","run_date","gold_discrep_unsettled_success_count"]

gold_discrep_unsettled_success_count_df = spark.createDataFrame([data],cols)

gold_discrep_unsettled_success_count_df.createOrReplaceTempView("gold_discrep_unsettled_success_count_temp")


spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING gold_discrep_unsettled_success_count_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.gold_discrep_unsettled_success_count=s.gold_discrep_unsettled_success_count
          WHEN NOT MATCHED then INSERT (run_id,run_date,gold_discrep_unsettled_success_count) VALUES(s.run_id,s.run_date,s.gold_discrep_unsettled_success_count)
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
gold_discrep_stale_pending_count=reconciled_df.filter(col("reconciliation")=="DISCREPANCY STALE PENDING").count()
data =[run_id,run_date,gold_discrep_stale_pending_count]
cols=["run_id","run_date","gold_discrep_stale_pending_count"]

gold_discrep_stale_pending_count_df = spark.createDataFrame([data],cols)
gold_discrep_stale_pending_count_df.createOrReplaceTempView("gold_discrep_stale_pending_count_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING gold_discrep_stale_pending_count_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.gold_discrep_stale_pending_count=s.gold_discrep_stale_pending_count
          WHEN NOT MATCHED then INSERT (run_id,run_date,gold_discrep_stale_pending_count) VALUES(s.run_id,s.run_date,s.gold_discrep_stale_pending_count)
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
orphan_record_count=reconciled_df.filter(col("reconciliation")=="ORPHAN RECORD").count()
data =[run_id,run_date,orphan_record_count]
cols=["run_id","run_date","orphan_record_count"]
orphan_record_count_df = spark.createDataFrame([data],cols)
orphan_record_count_df.createOrReplaceTempView("orphan_record_count_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING orphan_record_count_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.orphan_record_count=s.orphan_record_count
          WHEN NOT MATCHED then INSERT (run_id,run_date,orphan_record_count) VALUES(s.run_id,s.run_date,s.orphan_record_count)
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
run_id = dbutils.widgets.get("run_id")
run_date = dbutils.widgets.get("run_date")
if gold_count>0:
    discrepancy_rate=(gold_discrep_unsettled_success_count+gold_discrep_stale_pending_count+orphan_record_count)/gold_count
else:
    discrepancy_rate=0
data =[run_id,run_date,discrepancy_rate]
cols=["run_id","run_date","discrepancy_rate"]

discrepancy_rate_df = spark.createDataFrame([data],cols)
discrepancy_rate_df.createOrReplaceTempView("discrepancy_rate_temp")

spark.sql("""
          MERGE INTO workspace.upi_schema.upi_pipeline_metrics t
          USING discrepancy_rate_temp s
          on t.run_id=s.run_id
          WHEN MATCHED THEN UPDATE SET t.discrepancy_rate=s.discrepancy_rate
          WHEN NOT MATCHED then INSERT (run_id,run_date,discrepancy_rate) VALUES(s.run_id,s.run_date,s.discrepancy_rate)
          """)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]